# Lab — Computational Graphs and Backpropagation


In this exercise you will implement a small reverse-mode automatic differentiation engine for the function

$f(x,y)=(xy+x)^2$.

You will practice the **forward pass**, the **chain rule**, and **gradient accumulation** when a variable is reused.

Use only standard Python (no automatic differentiation libraries). Some helper code is provided to keep the exercise short.

At $x=2$ and $y=3$, the expected results are $f=64$, $\partial f/\partial x=64$, and $\partial f/\partial y=32$.

## 1. Create the computational graph

Introduce $a=xy$, $b=a+x$, and $f=b^2$.

```text
x ───────┐
│        │
├──► (*) a ──► (+) b ──► square ──► f
│      ▲          ▲
└──────┼──────────┘
       y
```

A node stores its operation, its parent nodes, its numerical value, and its gradient. **The class is already implemented**. Your task is to create the five nodes.

In [ ]:
class Node:
    def __init__(self, op, parents=None, value=None, name=None):
        self.op = op
        self.parents = parents or []
        self.value = value
        self.grad = 0.0
        self.name = name or op

    def __repr__(self):
        return f"Node({self.name}, value={self.value}, grad={self.grad})"


# TODO: create input nodes x, y and operation nodes a, b, f
# x = Node("input", value=2.0, name="x")
# y = ...
# a = ...
# b = ...
# f = ...

## 2. Forward pass

Implement `forward(node)` using **recursion**. First evaluate the parents, then apply the operation:

- `input`: return the stored value;
- `add`: sum the two parent values;
- `mul`: multiply the two parent values;
- `square`: square the parent value.

Store the result in `node.value`. Expected output: **64**.

For this small graph, repeated evaluation is acceptable; you do not need to implement memoization.

In [ ]:
def forward(node):
    if node.op == "input":
        return node.value

    values = [forward(parent) for parent in node.parents]

    if node.op == "add":
        node.value = ...  # TODO
    elif node.op == "mul":
        node.value = ...  # TODO
    elif node.op == "square":
        node.value = ...  # TODO
    else:
        raise ValueError(f"Unknown operation: {node.op}")

    return node.value


# Uncomment after constructing the graph:
# print("f =", forward(f))

## 3. Backpropagation

We want $\partial f/\partial x$ and $\partial f/\partial y$.

The local derivatives are:

- If $z=u+v$, then $\partial z/\partial u=1$ and $\partial z/\partial v=1$.
- If $z=uv$, then $\partial z/\partial u=v$ and $\partial z/\partial v=u$.
- If $z=u^2$, then $dz/du=2u$.

Start from $\partial f/\partial f=1$ and propagate gradients **backward**. The helper below already returns nodes in forward topological order (parents before children). Complete the missing gradient updates in `backward`.

**Important:** use `+=`, not `=`, because `x` contributes to $f$ through two paths.

In [ ]:
def topological_order(output):
    """Return every reachable node once, parents before children."""
    order = []
    visited = set()

    def visit(node):
        if node in visited:
            return
        visited.add(node)
        for parent in node.parents:
            visit(parent)
        order.append(node)

    visit(output)
    return order


def backward(output):
    nodes = topological_order(output)
    for node in nodes:
        node.grad = 0.0
    output.grad = 1.0

    for node in reversed(nodes):
        if node.op == "add":
            u, v = node.parents
            u.grad += ...  # TODO
            v.grad += ...  # TODO
        elif node.op == "mul":
            u, v = node.parents
            u.grad += ...  # TODO: use v.value
            v.grad += ...  # TODO: use u.value
        elif node.op == "square":
            u = node.parents[0]
            u.grad += ...  # TODO


# Uncomment after forward(f):
# backward(f)
# print("df/dx =", x.grad)  # expected 64
# print("df/dy =", y.grad)  # expected 32

## Check your results

After completing the exercises, run the following assertions. They should all pass.

In [ ]:
# Uncomment once all TODOs are completed:
# assert forward(f) == 64.0
# backward(f)
# assert abs(x.grad - 64.0) < 1e-9
# assert abs(y.grad - 32.0) < 1e-9
# print("All checks passed!")